In [ ]:
import pandas as pd
import os
import json
import numpy as np
import re
from tqdm.notebook import tqdm

In [ ]:
def loadJsonData(path_to_data: str) -> list[dict]:
    data = []
    
    for file_name in os.listdir(path_to_data):
        file_path = os.path.join(path_to_data, file_name)
        
        # Проверяем, что это файл (а не папка)
        if os.path.isfile(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                data.append(json.load(f))
                
    return data

In [ ]:
def make_base_dict():
    parsed_dict = {}
    parsed_dict.setdefault('uri', [])
    parsed_dict.setdefault('content', [])
    parsed_dict.setdefault('range', [])
    parsed_dict.setdefault('type', [])
    parsed_dict.setdefault('comments', [])

    return parsed_dict

def project_parser(project, pars_type : str, content : str, comments : list[str], parsed_data : dict): #pars_type == 'statements' or 'arguments'
    for txt in project['nodes'][pars_type]:
        uri = txt['uri']
        if uri not in parsed_data['uri']:
            parsed_data['uri'].append(txt['uri'])
            parsed_data['type'].append(pars_type)
            if 'range' in txt:
                parsed_data['range'].append(txt['range'])
            else:
                # print(f'{parsed_data['uri']} no range')
                parsed_data['range'].append(None)
            parsed_data['comments'].append('\n\n'.join([f'comment {i}: {comments[i]}' for i in range(len(comments))]))
            if 'com' in txt:
                num = txt['com']
                parsed_data['content'].append(comments[num])
            else:    
                parsed_data['content'].append(content)

    return parsed_data

def statement_check(project, arg, argument, arg_df, base_df, param):
    #check does uri has link to the data text and put the text from base_df to arg_df
    statement_uri = arg['params'][param]['param']
    if arg['params'][param]['is_statement'] == True:
        statement_df = base_df[base_df['uri'] == statement_uri]
        txt = []
        for i, rng in enumerate(statement_df['range'].item()):
            if i > 0:
                 txt.append('<del>')
            txt.append(statement_df['content'].item()[rng[0]:rng[1]])
        return ' '.join(txt)
        #-----------------------
        # arg_df.loc[arg_df['uri'] == argument, 'content'] = statement_df['content'].item()
        #-----------------------
    else: 
        arg_df.loc[arg_df['uri'] == argument, 'attacked_arg_uri'] = statement_uri
        in_arg = project['data']['arguments'][statement_uri]

        premises = []
        conclusion = []

        for in_param in in_arg['params']:
            if in_param == 'hasConclusion':
                conclusion.append(statement_check(project, in_arg, argument, arg_df, base_df, in_param))
            elif in_param.split('_')[-1] == 'Premise':
                premises.append(statement_check(project, in_arg, argument, arg_df, base_df, in_param))

        #flatten lists
        def flatten_and_clean(data):
            for sublist in data:
                for item in (sublist if isinstance(sublist, list) else [sublist]):
                    cleaned = str(item).strip()
                    if cleaned:  # пропускаем пустые строки
                        yield cleaned

        premises_clean = flatten_and_clean(premises)
        conclusion_clean = flatten_and_clean(conclusion)

        result = ' '.join(list(premises_clean) + list(conclusion_clean))
        return result
        


def parseData(data : dict) -> dict:
    assert data, "no data"

    conflict_df = pd.DataFrame()
    inference_df = pd.DataFrame()

    for subcorpora in tqdm(data['subcorpora']):
        texts = subcorpora['texts'] 
        if texts:
            for text in texts:
                content = text['text']['content']
                comments = [comment['content'] for comment in text['text']['comments']] if 'comments' in text['text'] else []
                for project in text['projects']:
                    base_dict = make_base_dict()
                    base_dict = project_parser(project, 'statements', content, comments, base_dict)
                    base_dict = project_parser(project, 'arguments', content, comments, base_dict)
                    #now work with df
                    base_df = pd.DataFrame(base_dict) #dict of all texts and uri
                    # print(f'for project "{project['id']}" duplicates: {base_df['uri'].duplicated().any()}')
                    #process arguments 

                    #arguments: argument - uri, arg - link on dict
                    for argument in project['data']['arguments']:
                        arg = project['data']['arguments'][argument]
                        arg_df = pd.DataFrame({'uri' : argument}, index=[0])
                        scheme_df = pd.DataFrame({'scheme' : arg['scheme'].split('#')[-1]}, index=[0])
                        content_df = pd.DataFrame({'content' : base_df[base_df['uri'] == argument]['content'].item()}, index=[0])
                        comments_df = pd.DataFrame({'comments' : base_df[base_df['uri'] == argument]['comments'].item()}, index=[0])
                        arg_df = pd.concat([arg_df, scheme_df, content_df, comments_df], axis=1)

                        params = [it for it in arg['params']]

                        arg_type = arg_df['scheme'].item().split('_')[-1]
                        if arg_type == 'Conflict':
                            exception_count = 0
                            i = 0
                            for param in params:
                                if param.split('_')[-1] == 'Exception':
                                    # statement_check(project, arg, argument, arg_df, base_df, param, column=f'Exception_{exception_count}')
                                    arg_df.loc[arg_df['uri'] == argument, f'Conclusion'] = statement_check(
                                        project, arg, argument, arg_df, base_df, param)
                                    exception_count += 1
                                else:
                                    # statement_check(project, arg, argument, arg_df, base_df, param, column=f'ConflictElement_{i}')
                                    arg_df.loc[arg_df['uri'] == argument, f'Premise_{i}'] = statement_check(
                                        project, arg, argument, arg_df, base_df, param)
                                    i += 1
                            conflict_df = pd.concat([conflict_df, arg_df], ignore_index=True)

                        elif arg_type == 'Inference':
                            premise_count = 0
                            for param in params:
                                if param.split('_')[-1] == 'Premise':
                                    arg_df.loc[arg_df['uri'] == argument, f'Premise_{premise_count}'] = statement_check(
                                        project, arg, argument, arg_df, base_df, param)
                                    premise_count += 1
                                else:
                                    arg_df.loc[arg_df['uri'] == argument, f'Conclusion'] = statement_check(
                                        project, arg, argument, arg_df, base_df, param)
                            inference_df = pd.concat([inference_df, arg_df], ignore_index=True)
                            
                        # parsed_df = pd.concat([parsed_df, arg_df], ignore_index=True)
        else:
            in_inf_df, in_conf_df = parseData(subcorpora)
            inference_df = pd.concat([inference_df, in_inf_df], ignore_index=True)
            conflict_df = pd.concat([conflict_df, in_conf_df], ignore_index=True)

    return inference_df, conflict_df

Load all files from data folder

In [ ]:
data_colletction = loadJsonData("./data")

Parse all data files

In [ ]:
parsed_data = [parseData(data) for data in data_colletction]

In [ ]:
parsed_data = [[x.fillna('NaN') for x in d] for d in parsed_data]
parsed_data = [[x[np.sort(x.keys())] for x in d] for d in parsed_data]

In [ ]:
train_inf_data = parsed_data[0][0]
train_conf_data = parsed_data[0][1]

In [ ]:
train_inf_data.to_csv('inf_8.csv', index=False)
train_conf_data.to_csv('conf_8.csv', index=False)

In [ ]:
parsed_data = [[x.fillna('NaN') for x in d] for d in parsed_data]
parsed_data = [[x[np.sort(x.keys())] for x in d] for d in parsed_data]
inf_data = pd.concat([inf for inf, _ in parsed_data], ignore_index=True)
conf_data = pd.concat([conf for _, conf in parsed_data], ignore_index=True)

print(f"len inf = {len(inf_data)}\nlen conf = {len(conf_data)}")